# 01 — Data Loading & ExplorationLoad GTEx data, filter to Whole Blood, build expression matrix, run PCA.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom sklearn.decomposition import PCAfrom sklearn.preprocessing import StandardScalerfrom gtex_biomarkers.config import Configfrom gtex_biomarkers.data import (    load_raw_data, filter_whole_blood,    build_blood_expression_matrix, build_blood_subjid,)Config.ensure_dirs()

## Load Input Files

In [ ]:
df_expr, df_samples, df_age, df_meta_url = load_raw_data()print("Expression shape:", df_expr.shape)print("Sample metadata:", df_samples.shape)print("Donor metadata:", df_age.shape)print("Pathology metadata:", df_meta_url.shape)

## Explore Loaded DataFrames

In [ ]:
df_expr.head()

In [ ]:
df_samples

In [ ]:
df_age

In [ ]:
df_meta_url

## Filter to Whole Blood & Build Expression Matrix

In [ ]:
blood_meta = filter_whole_blood(df_samples)print("Whole Blood samples:", blood_meta["SAMPID"].nunique())X_wb, df_blood_wb = build_blood_expression_matrix(df_expr, blood_meta)print("X_wb shape (samples × genes):", X_wb.shape)blood_subjid = build_blood_subjid(X_wb)X_wb.head()

## PCA on Whole Blood Expression

In [ ]:
X_scaled = StandardScaler().fit_transform(X_wb)pca = PCA(n_components=10, random_state=0)pcs = pca.fit_transform(X_scaled)print("Explained variance (PC1-10):", pca.explained_variance_ratio_)

### Cumulative Explained Variance

In [ ]:
var = pca.explained_variance_ratio_cum_var = np.cumsum(var)plt.figure(figsize=(8, 5))plt.plot(range(1, len(cum_var) + 1), cum_var, marker="o")plt.title("Cumulative Explained Variance — Whole Blood PCA")plt.xlabel("PC"); plt.ylabel("Cumulative variance"); plt.grid(True)plt.savefig(Config.FIGURES_DIR / "pca_whole_blood_cumulative_explained_variance.pdf",            dpi=300, bbox_inches="tight")plt.show()

### PC1 vs PC2 — Colored by Sex

In [ ]:
pc_cols = [f"PC{i}" for i in range(1, pcs.shape[1] + 1)]pca_scores = pd.DataFrame(pcs, columns=pc_cols)pca_scores["SAMPID"] = X_wb.indexpca_scores["SUBJID"] = pca_scores["SAMPID"].astype(str).str.split("-").str[:2].str.join("-")df_meta_url["SUBJID"] = df_meta_url["Tissue.Sample.ID"].astype(str).str.split("-").str[:2].str.join("-")sex_df = df_meta_url[["SUBJID", "Sex"]].dropna().drop_duplicates("SUBJID")plot_df = pca_scores.merge(sex_df, on="SUBJID", how="left")if plot_df["Sex"].notna().any():    plt.figure(figsize=(9, 6))    for s, sub in plot_df.groupby("Sex"):        plt.scatter(sub["PC1"], sub["PC2"], alpha=0.5, s=10, label=f"SEX={s}")    plt.title("PCA of Whole Blood (PC1 vs PC2) colored by SEX")    plt.xlabel("PC1"); plt.ylabel("PC2"); plt.grid(True); plt.legend()    plt.savefig(Config.FIGURES_DIR / "pca_whole_blood_pc1_pc2_by_sex.pdf",                dpi=300, bbox_inches="tight")    plt.show()